# Computer Prices 2025 — Notebook de projet

Ce notebook accompagne le travail sur la comp?tition Kaggle [Computer Prices 2025](https://www.kaggle.com/competitions/computer-prices-2025/data). Les fichiers de la comp?tition sont stock?s localement dans `computer-prices-2025/` ? la racine du projet. Le notebook est organis? en trois ?tapes : exploration des donn?es, mod?les de r?f?rence et ensembles, puis am?lioration et analyse des mod?les.

## 1. Data Exploratory Analysis

L’analyse exploratoire permet de comprendre la structure du jeu de données avant toute modélisation. Nous allons charger les fichiers d’entraînement et de test, examiner leurs dimensions, leurs variables et leurs types, puis mesurer les statistiques descriptives, les valeurs manquantes, les doublons et la cardinalité des colonnes. Si la colonne cible peut être déduite sans ambiguïté des fichiers, nous en examinerons également la distribution. Cette étape sert à repérer les particularités qui guideront le prétraitement, sans supposer à l’avance le nom ou la nature des variables.

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path.cwd() / "computer-prices-2025"
train_path = DATA_DIR / "computer_prices_all.csv"
test_path = DATA_DIR / "computer_prices_test.csv"
submission_path = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = (
    pd.read_csv(submission_path) if submission_path.is_file() else None
)

print(f"Dossier des donn?es : {DATA_DIR}")
print(f"Fichier d'entra?nement : {train_path}")
print(f"Fichier de test         : {test_path}")
print(
    "Fichier de soumission   : "
    f"{submission_path if sample_submission is not None else 'non fourni'}"
)

Dossier des donn?es : c:\Users\cypri\OneDrive\Desktop\IMT\LOGIN\AAA\Laptop-Price\computer-prices-2025
Fichier d'entra?nement : c:\Users\cypri\OneDrive\Desktop\IMT\LOGIN\AAA\Laptop-Price\computer-prices-2025\computer_prices_all.csv
Fichier de test         : c:\Users\cypri\OneDrive\Desktop\IMT\LOGIN\AAA\Laptop-Price\computer-prices-2025\computer_prices_test.csv
Fichier de soumission   : c:\Users\cypri\OneDrive\Desktop\IMT\LOGIN\AAA\Laptop-Price\computer-prices-2025\sample_submission.csv


### Dimensions et aperçu des données

On commence par vérifier le nombre d’observations et de colonnes disponibles, puis par afficher quelques lignes de chaque jeu.

In [4]:
print(f"Train : {train.shape[0]:,} lignes × {train.shape[1]} colonnes")
print(f"Test  : {test.shape[0]:,} lignes × {test.shape[1]} colonnes")

display(train.head())
display(test.head())

Train : 100,000 lignes × 34 colonnes
Test  : 50,000 lignes × 33 colonnes


,ID,device_type,brand,model,release_year,os,form_factor,cpu_brand,cpu_model,cpu_tier,cpu_cores,cpu_threads,cpu_base_ghz,cpu_boost_ghz,gpu_brand,gpu_model,gpu_tier,vram_gb,ram_gb,storage_type,storage_gb,storage_drive_count,display_type,display_size_in,resolution,refresh_hz,battery_wh,charger_watts,psu_watts,wifi,bluetooth,weight_kg,warranty_months,price
0,0,Desktop,Samsung,Samsung Forge XDI,2022,Windows,ATX,Intel,Intel i5-11129,3,12,24,2.8,3.8,NVIDIA,RTX 40 60,2,6,16,NVMe,1024,1,LED,27.0,2560x1440,90,0,0,750,Wi-Fi 6,5.1,11.00,36,1383.99
1,1,Laptop,Samsung,Samsung Pro KM8,2022,Windows,Mainstream,Intel,Intel i7-11114,4,12,24,2.6,3.6,NVIDIA,RTX 40 80,4,10,64,NVMe,512,1,OLED,16.0,1920x1080,90,56,120,0,Wi-Fi 6,5.3,2.03,12,2274.99
2,2,Desktop,Lenovo,Lenovo Strix BIE,2024,macOS,SFF,AMD,AMD Ryzen 5 5168,2,8,16,2.6,3.6,NVIDIA,RTX 40 50,1,4,8,NVMe,512,2,LED,32.0,3440x1440,120,0,0,850,Wi-Fi 6,5.0,7.00,24,1879.99
3,3,Desktop,Dell,Dell Cube AXR,2024,Windows,ATX,AMD,AMD Ryzen 5 7550,2,6,12,2.6,3.6,AMD,RX 7000 60,2,6,16,HDD,512,2,IPS,27.0,3440x1440,120,0,0,650,Wi-Fi 6,5.2,6.00,36,1331.99
4,4,Laptop,Gigabyte,Gigabyte Pro IX1,2024,Linux,Gaming,AMD,AMD Ryzen 7 6230,5,16,32,2.8,3.9,NVIDIA,RTX 30 80 Ti,5,12,96,NVMe,256,1,Mini-LED,15.6,2560x1600,90,80,90,0,Wi-Fi 6,5.2,1.50,12,2681.99


,ID,device_type,brand,model,release_year,os,form_factor,cpu_brand,cpu_model,cpu_tier,cpu_cores,cpu_threads,cpu_base_ghz,cpu_boost_ghz,gpu_brand,gpu_model,gpu_tier,vram_gb,ram_gb,storage_type,storage_gb,storage_drive_count,display_type,display_size_in,resolution,refresh_hz,battery_wh,charger_watts,psu_watts,wifi,bluetooth,weight_kg,warranty_months
0,100000,Laptop,Lenovo,Lenovo Zen ZTR,2020,macOS,Mainstream,Intel,Intel i7-10922,5,16,32,2.8,3.8,NVIDIA,RTX 30 80 Ti,5,12,96,NVMe,256,1,Mini-LED,15.6,2880x1800,120,80,65,0,Wi-Fi 6,5.0,1.50,24
1,100001,Laptop,MSI,MSI Zen MZA,2019,Windows,Gaming,Intel,Intel i7-11210,4,12,22,2.6,3.7,Intel,Arc A700,4,0,64,Hybrid,1024,1,OLED,16.0,2560x1600,90,56,90,0,Wi-Fi 6,5.1,1.53,12
2,100002,Desktop,MSI,MSI Pro 3IT,2020,Windows,Micro-ATX,Intel,Intel i5-12687,2,6,10,2.6,3.6,NVIDIA,RTX 30 60,2,6,24,HDD,256,2,LED,24.0,3440x1440,60,0,0,550,Wi-Fi 7,5.2,6.00,12
3,100003,Desktop,Samsung,Samsung Pro 9A8,2025,Windows,Micro-ATX,Intel,Intel i5-11832,2,8,16,2.6,3.5,Intel,Arc A380,2,2,16,SSD,256,4,Mini-LED,24.0,1920x1080,165,0,0,750,Wi-Fi 6,5.3,9.00,36
4,100004,Laptop,Apple,Apple Nitro 00I,2022,Linux,Ultrabook,Apple,Apple M1,6,24,24,3.0,4.0,Apple,Apple Integrated,6,0,128,HDD,1024,1,LED,15.6,2560x1600,60,90,65,0,Wi-Fi 5,4.2,2.50,24


### Variables et types

Cette vue recense les colonnes d’entraînement et leur type pandas. Les colonnes absentes du jeu de test sont signalées séparément, car elles peuvent inclure la cible.

In [8]:
feature_overview = pd.DataFrame({
    "feature": train.columns,
    "dtype": train.dtypes.astype(str).values,
    "present_in_test": train.columns.isin(test.columns),
})
display(feature_overview)

train_only_columns = [column for column in train.columns if column not in test.columns]
print("Colonnes présentes dans train mais absentes de test :", train_only_columns or "aucune")

,feature,dtype,present_in_test
0,ID,int64,True
1,device_type,str,True
2,brand,str,True
3,model,str,True
4,release_year,int64,True
5,os,str,True
6,form_factor,str,True
7,cpu_brand,str,True
8,cpu_model,str,True
9,cpu_tier,int64,True


Colonnes présentes dans train mais absentes de test : ['price']


### Statistiques descriptives

`describe` fournit des statistiques adaptées aux colonnes numériques et catégorielles. Les résultats sont transposés pour faciliter la comparaison entre variables.

In [19]:
#display(train.describe(include="all").T)
df = train.copy()
cpu_check = (
    df.groupby("cpu_model")[cols]
      .nunique()
      .max(axis=1)
      .sort_values(ascending=False)
)

model = cpu_check.index[0]

print("CPU suspect :", model)

print(
    df.loc[df["cpu_model"] == model, ["cpu_model"] + cols]
      .drop_duplicates()
      .sort_values(cols)
      .to_string(index=False)
)

CPU suspect : Apple M1 Pro
   cpu_model  cpu_tier  cpu_cores  cpu_threads  cpu_base_ghz  cpu_boost_ghz
Apple M1 Pro         1          4            4           2.0            2.8
Apple M1 Pro         1          4            4           2.0            2.9
Apple M1 Pro         1          4            4           2.0            3.0
Apple M1 Pro         1          4            4           2.0            3.1
Apple M1 Pro         1          4            4           2.4            3.2
Apple M1 Pro         1          4            4           2.4            3.3
Apple M1 Pro         1          4            4           2.4            3.4
Apple M1 Pro         1          4            4           2.4            3.5
Apple M1 Pro         1          6            6           2.0            2.8
Apple M1 Pro         1          6            6           2.0            2.9
Apple M1 Pro         1          6            6           2.0            3.0
Apple M1 Pro         1          6            6           2.0 

### Valeurs manquantes, doublons et cardinalité

Le tableau ci-dessous présente le nombre et le pourcentage de valeurs manquantes ainsi que le nombre de valeurs distinctes par colonne. Le nombre de lignes dupliquées est affiché séparément.

In [7]:
missing_summary = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_percent": train.isna().mean().mul(100).round(2),
    "distinct_values": train.nunique(dropna=True),
}).sort_values(["missing_count", "distinct_values"], ascending=[False, True])

display(missing_summary)
print(f"Lignes dupliquées dans train : {train.duplicated().sum():,}")
print(f"Lignes dupliquées dans test  : {test.duplicated().sum():,}")

,missing_count,missing_percent,distinct_values
device_type,0,0.0,2
cpu_brand,0,0.0,3
os,0,0.0,4
gpu_brand,0,0.0,4
storage_type,0,0.0,4
storage_drive_count,0,0.0,4
wifi,0,0.0,4
warranty_months,0,0.0,4
storage_gb,0,0.0,5
bluetooth,0,0.0,5


Lignes dupliquées dans train : 0
Lignes dupliquées dans test  : 0


### Résumés par type de variable

Les colonnes numériques et non numériques sont résumées séparément. La catégorie « non numérique » inclut notamment les colonnes de type texte, booléen ou catégoriel reconnues par pandas.

In [ ]:
numeric_columns = train.select_dtypes(include="number").columns.tolist()
categorical_columns = train.select_dtypes(exclude="number").columns.tolist()

print(f"Variables numériques ({len(numeric_columns)}) : {numeric_columns}")
print(f"Variables non numériques ({len(categorical_columns)}) : {categorical_columns}")

if numeric_columns:
    display(train[numeric_columns].describe().T)
if categorical_columns:
    display(train[categorical_columns].describe().T)

### Fichier de soumission et cible potentielle

Le fichier d’exemple, lorsqu’il est disponible, renseigne sur le format attendu pour les prédictions. Une colonne cible n’est proposée que si une unique colonne du jeu d’entraînement est absente du jeu de test ; dans le cas contraire, le notebook ne devine pas laquelle prédire.

In [ ]:
if sample_submission is not None:
    print(f"Dimensions du fichier de soumission : {sample_submission.shape}")
    display(sample_submission.head())
else:
    print("sample_submission.csv n'a pas été trouvé ; cette vérification est ignorée.")

if len(train_only_columns) == 1:
    target_column = train_only_columns[0]
    print(f"Cible potentielle (unique) : {target_column!r}")
    if pd.api.types.is_numeric_dtype(train[target_column]):
        display(train[target_column].describe().to_frame("statistiques_cible"))
    else:
        display(train[target_column].value_counts(dropna=False).head(20).to_frame("effectif"))
elif train_only_columns:
    target_column = None
    print("Plusieurs colonnes candidates à la cible ; aucune n'est sélectionnée automatiquement.")
else:
    target_column = None
    print("Aucune cible n'a pu être identifiée par comparaison des colonnes train/test.")

### Conclusion de l’analyse exploratoire

Les tableaux précédents permettent de caractériser la taille des données, les types de variables, leurs distributions, les valeurs manquantes, les doublons et les colonnes à forte cardinalité. Il faut relever ici les constats réellement observés après exécution : quelles variables nécessitent un traitement des valeurs manquantes ou des catégories, si des doublons sont présents, et si la cible a pu être identifiée sans ambiguïté. Ces éléments serviront à définir un prétraitement cohérent et à éviter toute fuite de données lors de la modélisation.

## 2. ML Baseline & Ensemble Models

Cette partie établira une référence de performance simple avant d’évaluer des modèles ensemblistes. Une séparation adaptée des données, une métrique conforme à la compétition et un prétraitement ajusté aux variables observées permettront de comparer les approches de manière fiable. Les résultats devront être interprétés par rapport à cette référence, sans utiliser le jeu de test pour entraîner ou choisir les modèles.

### Conclusion de la partie

La conclusion comparera les scores de validation de la baseline et des modèles ensemblistes, la stabilité de leurs performances et les principaux compromis observés. Elle précisera quel modèle constitue la meilleure référence pour la suite et les limites qui motivent une optimisation supplémentaire.

## 3. Enhanced Models, Hyperparameter Tuning & Analysis

Cette dernière partie visera à améliorer les modèles retenus à l’étape précédente au moyen d’un réglage raisonné des hyperparamètres et d’analyses complémentaires. Les choix devront être évalués avec une stratégie de validation cohérente et comparés aux résultats de référence afin de vérifier que les gains sont robustes et pertinents pour la compétition.

### Conclusion de la partie

La conclusion présentera les performances finales et leur comparaison avec les références, les paramètres ou choix méthodologiques qui ont eu le plus d’impact, ainsi que les limites restantes. Elle résumera le modèle et la stratégie de validation retenus avant toute prédiction finale ou soumission.